In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.45.0", "peft==0.13.0", "accelerate==0.34.2",
    "datasets==3.0.1", "trl==0.11.1", "qwen-vl-utils", "Pillow",
    "rouge-score", "sacrebleu", "bitsandbytes", "matplotlib", "numpy",
])

import os, json, random, warnings, textwrap
from io import BytesIO
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from PIL import Image, ImageDraw
import torch
from torch.utils.data import Dataset

from datasets import load_dataset
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU

warnings.filterwarnings("ignore")

# ============================================
# CELL 2: Configuration & Dual GPU Setup
# ============================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
    if torch.cuda.device_count() >= 2:
        print(f"Using {torch.cuda.device_count()} GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
        os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    else:
        print(f"Only {torch.cuda.device_count()} GPU(s) available")

CFG = {
    "model_id"                    : "Qwen/Qwen2-VL-2B-Instruct",
    "output_dir"                  : "/kaggle/working/vlm_qlora_output",
    "final_model_dir"             : "/kaggle/working/final_saved_model",
    "comparison_dir"              : "/kaggle/working/comparisons",
    "epochs_comparison_dir"       : "/kaggle/working/epochs_comparison",
    "checkpoints_dir"             : "/kaggle/working/checkpoints",
    "lora_r"                      : 16,
    "lora_alpha"                  : 32,
    "lora_dropout"                : 0.05,
    "lora_target_modules"         : ["q_proj", "v_proj", "k_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    "num_train_epochs"            : 5,
    "per_device_train_batch_size" : 1,
    "per_device_eval_batch_size"  : 1,
    "gradient_accumulation_steps" : 4,
    "learning_rate"               : 1e-4,
    "warmup_ratio"                : 0.10,
    "lr_scheduler_type"           : "cosine",
    "optim"                       : "paged_adamw_8bit",
    "bf16"                        : True,
    "fp16"                        : False,
    "logging_steps"               : 10,
    "eval_strategy"               : "steps",
    "eval_steps"                  : 50,
    "save_steps"                  : 50,
    "save_total_limit"            : 3,
    "load_best_model_at_end"      : True,
    "metric_for_best_model"       : "eval_loss",
    "dataloader_num_workers"      : 0,
    "max_samples"                 : 4000,  # Keep at 4000 for dataset loading
    "train_ratio"                 : 0.80,
    "image_size"                  : 768,
    "max_seq_len"                 : 2048,
    "quantization_type"           : "nf4",
    "compute_dtype"               : torch.bfloat16,
}

for d in [CFG["output_dir"], CFG["final_model_dir"], CFG["comparison_dir"], 
          CFG["epochs_comparison_dir"], CFG["checkpoints_dir"]]:
    os.makedirs(d, exist_ok=True)

print(f"Configuration loaded with {CFG['num_train_epochs']} epochs")
print(f"Max samples: {CFG['max_samples']}")

# ============================================
# CELL 3: Helper Functions
# ============================================
def safe_text(text):
    if text is None: return ""
    if isinstance(text, (int, float)): return str(text)
    if isinstance(text, bytes): return text.decode("utf-8", errors="ignore")
    if isinstance(text, list): return " ".join(str(x) for x in text)
    return str(text)

def resize_image(img, target=None):
    target = target or CFG["image_size"]
    try:
        if img is None: return None
        if isinstance(img, dict): img = img.get("bytes", None)
        if img is None: return None
        if not isinstance(img, Image.Image):
            if isinstance(img, np.ndarray): img = Image.fromarray(img)
            elif isinstance(img, bytes): img = Image.open(BytesIO(img))
            else: return None
        w, h = img.size
        if min(w, h) <= target: return img
        scale = target / min(w, h)
        return img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    except Exception as e:
        print(f"resize_image error: {e}")
        return None

def validate_image(img):
    try:
        if img is None or isinstance(img, dict): return False
        if not isinstance(img, Image.Image): img = Image.fromarray(np.array(img))
        img.convert("RGB")
        return True
    except:
        return False

# ============================================
# CELL 4: Dataset Loading & Exploration
# ============================================
print("\n" + "="*50)
print("Dataset Exploration")
print("="*50)

dataset_path = "/kaggle/input/datasets/zphilip/nougat-training-dataset-example"

try:
    from datasets import Dataset, Image as ImageFeature
    
    print("Loading images and associated .mmd files from directory structure...")
    
    image_paths = []
    text_contents = []
    skipped_no_mmd = 0
    skipped_short_text = 0
    
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                mmd_path = img_path.replace('.png', '.mmd').replace('.jpg', '.mmd').replace('.jpeg', '.mmd')
                
                if os.path.exists(mmd_path):
                    try:
                        with open(mmd_path, 'r', encoding='utf-8') as f:
                            text_content = f.read()
                        if len(text_content.strip()) > 10:
                            image_paths.append(img_path)
                            text_contents.append(text_content)
                        else:
                            skipped_short_text += 1
                    except Exception as e:
                        print(f"Error reading {mmd_path}: {e}")
                else:
                    skipped_no_mmd += 1
    
    print(f"Found {len(image_paths)} valid image-mmd pairs")
    print(f"   Skipped {skipped_no_mmd} images with no matching .mmd file")
    print(f"   Skipped {skipped_short_text} images with text too short (<10 chars)")
    
    if len(image_paths) > 0:
        data_dict = {"image": image_paths, "text": text_contents}
        raw_ds = Dataset.from_dict(data_dict)
        raw_ds = raw_ds.cast_column("image", ImageFeature())
        TEXT_COL = "text"
        print(f"\nDataset loaded successfully")
        print(f"   Total samples: {len(raw_ds)}")
        print(f"   Using text column: '{TEXT_COL}'")
        
        sample = raw_ds[0]
        print(f"\nSample Exploration:")
        print(f"   Image size: {sample['image'].size}")
        text_sample = safe_text(sample[TEXT_COL])
        print(f"   Text length: {len(text_sample)} characters")
        print(f"   Text preview: {text_sample[:200]}...")
    else:
        raise ValueError("No valid image-mmd pairs found")
        
except Exception as e:
    print(f"Dataset load error: {e}, using synthetic data")
    from datasets import Dataset as HFDataset
    rows = []
    for i in range(CFG["max_samples"]):
        img = Image.new("RGB", (1024, 1024), "white")
        draw = ImageDraw.Draw(img)
        md = f"# Document {i}\n\n## Introduction\n\nSynthetic document.\n\n- Bullet A\n- Bullet B"
        draw.text((20, 20), md[:300], fill="black")
        rows.append({"image": img, "text": md})
    raw_ds = HFDataset.from_list(rows)
    TEXT_COL = "text"
    print(f"Synthetic dataset created with {len(raw_ds)} samples")

print("\nVisual Inspection (saving 3 samples):")
for i in range(min(3, len(raw_ds))):
    try:
        img = raw_ds[i]["image"]
        if isinstance(img, Image.Image):
            img.save(f"/kaggle/working/spot_check_{i}.png")
            print(f"   Sample {i}: saved spot_check_{i}.png")
    except Exception as e:
        print(f"   Sample {i} failed: {e}")

# ============================================
# CELL 5: Data Preparation (ChatML Format)
# ============================================
print("\n" + "="*50)
print("Data Preparation")
print("="*50)

PROMPT_STYLES = {
    "standard": "Convert this document image to Markdown format.",
    "detailed": (
        "You are an expert document parser. Convert the provided document image "
        "into well-structured Markdown. Preserve headings, paragraphs, lists, tables, "
        "code blocks, mathematical equations, and formatting. Output ONLY the Markdown content."
    ),
    "concise": "Extract all text and structure from this document. Output only Markdown.",
}

def build_chatml(image, markdown_text, prompt_style="detailed"):
    try:
        if not validate_image(image):
            return None
        markdown_text = safe_text(markdown_text)
        if len(markdown_text.strip()) < 10:
            return None
        img = resize_image(
            image.convert("RGB") if isinstance(image, Image.Image)
            else Image.fromarray(np.array(image)).convert("RGB"),
            CFG["image_size"]
        )
        if img is None:
            return None
        return {
            "messages": [
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": PROMPT_STYLES[prompt_style]},
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": markdown_text}
                ]},
            ],
            "image": img,
            "markdown": markdown_text,
        }
    except Exception as e:
        return None

subset_size = min(CFG["max_samples"], len(raw_ds))
chatml_samples = []
print(f"Processing {subset_size} samples...")

for i in range(subset_size):
    if i % 500 == 0 and i > 0:
        print(f"  Processed {i}/{subset_size} samples...")
    try:
        text_content = safe_text(raw_ds[i][TEXT_COL])
        if len(text_content.strip()) > 10:
            s = build_chatml(raw_ds[i]["image"], text_content, "detailed")
            if s:
                chatml_samples.append(s)
    except Exception as e:
        continue

if len(chatml_samples) < 100:
    print(f"Only {len(chatml_samples)} valid samples, generating synthetic data...")
    needed = CFG["max_samples"] - len(chatml_samples)
    for i in range(needed):
        try:
            img = Image.new("RGB", (1024, 1024), "white")
            draw = ImageDraw.Draw(img)
            md = f"# Synthetic Doc {i}\n\n## Section\n\nTest content.\n\n- Item A\n- Item B"
            draw.text((20, 20), md[:300], fill="black")
            s = build_chatml(img, md, "detailed")
            if s:
                chatml_samples.append(s)
            if len(chatml_samples) >= CFG["max_samples"]:
                break
        except:
            continue

if len(chatml_samples) == 0:
    raise ValueError("No valid training samples found")

print(f"Created {len(chatml_samples)} ChatML samples")

# ============================================
# CELL 6: Dataset Splitting WITH SAMPLE REDUCTION TO 160
# ============================================
print("\n" + "="*50)
print("Dataset Subsetting")
print("="*50)

random.shuffle(chatml_samples)
n = len(chatml_samples)
n_train = int(n * CFG["train_ratio"])
n_val = n - n_train
n_test = min(100, n_val // 2)
n_val = n_val - n_test

train_data = chatml_samples[:n_train]
val_data = chatml_samples[n_train:n_train+n_val]
test_data = chatml_samples[n_train+n_val:n_train+n_val+n_test]

print(f"Original Dataset Split:")
print(f"   Training:   {len(train_data)} samples ({len(train_data)/n*100:.1f}%)")
print(f"   Validation: {len(val_data)} samples ({len(val_data)/n*100:.1f}%)")
print(f"   Test:       {len(test_data)} samples ({len(test_data)/n*100:.1f}%)")

# ============================================
# REDUCE TRAINING SAMPLES TO 160
# ============================================
TARGET_TRAINING_SAMPLES = 160  # CHANGE THIS VALUE TO CONTROL TRAINING SIZE

print(f"\n{'='*50}")
print(f"Reducing training samples from {len(train_data)} to {TARGET_TRAINING_SAMPLES}")
print(f"{'='*50}")

if len(train_data) > TARGET_TRAINING_SAMPLES:
    train_data = train_data[:TARGET_TRAINING_SAMPLES]
    print(f"✓ Training samples: {len(train_data)}")
    
    # Adjust validation proportionally (about 20-30% of training)
    val_target = max(30, TARGET_TRAINING_SAMPLES // 5)
    if len(val_data) > val_target:
        val_data = val_data[:val_target]
    print(f"✓ Validation samples: {len(val_data)}")
    
    # Keep test samples manageable
    if len(test_data) > 40:
        test_data = test_data[:40]
    print(f"✓ Test samples: {len(test_data)}")
else:
    print(f"Training samples already at or below {TARGET_TRAINING_SAMPLES}")

print(f"\nFinal Dataset Split for Training:")
print(f"   Training:   {len(train_data)} samples")
print(f"   Validation: {len(val_data)} samples")
print(f"   Test:       {len(test_data)} samples")

# ============================================
# CELL 7: QLoRA Setup & Model Loading
# ============================================
print("\n" + "="*50)
print("QLoRA Fine-Tuning Setup")
print("="*50)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=CFG["quantization_type"],
    bnb_4bit_compute_dtype=CFG["compute_dtype"],
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(CFG["model_id"], trust_remote_code=True)
processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 1280 * 28 * 28

model = Qwen2VLForConditionalGeneration.from_pretrained(
    CFG["model_id"],
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=CFG["compute_dtype"],
    quantization_config=bnb_config,
)

print(f"Model loaded with quantization")
print(f"   Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ============================================
# CELL 8: LoRA Configuration
# ============================================
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["lora_target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\nModel Statistics:")
print(f"   Total parameters: {total:,}")
print(f"   Trainable parameters: {trainable:,}")
print(f"   Trainable percentage: {100 * trainable / total:.2f}%")

# ============================================
# CELL 9: Dataset Class & Collation
# ============================================
class MarkdownDataset(Dataset):
    def __init__(self, samples_list):
        self.samples_list = samples_list
    
    def __len__(self):
        return len(self.samples_list)
    
    def __getitem__(self, idx):
        if isinstance(idx, list):
            return [self.samples_list[i] for i in idx]
        return self.samples_list[idx]
    
    def __getitems__(self, indices):
        return [self.samples_list[i] for i in indices]

def custom_collate_fn(batch):
    if batch and isinstance(batch[0], list):
        batch = [item for sublist in batch for item in sublist]
    
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    
    msgs_list = []
    for item in batch:
        if isinstance(item, dict) and "messages" in item:
            msgs_list.append(item["messages"])
    
    if not msgs_list:
        return None
    
    try:
        texts = []
        for msgs in msgs_list:
            text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        
        images, videos = process_vision_info(msgs_list)
        
        inputs = processor(
            text=texts,
            images=images,
            videos=videos,
            padding=True,
            truncation=True,
            max_length=CFG["max_seq_len"],
            return_tensors="pt"
        )
        
        labels = inputs["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        
        assistant_start = "<|im_start|>assistant"
        assistant_ids = processor.tokenizer.encode(assistant_start, add_special_tokens=False)
        
        for i in range(len(labels)):
            for j in range(len(labels[i]) - len(assistant_ids)):
                if labels[i][j:j+len(assistant_ids)].tolist() == assistant_ids:
                    labels[i][:j+len(assistant_ids)] = -100
                    break
        
        inputs["labels"] = labels
        return inputs
        
    except Exception as e:
        print(f"Error in collate_fn: {e}")
        return None

# Create datasets
train_dataset = MarkdownDataset(train_data)
val_dataset = MarkdownDataset(val_data)

# ============================================
# CELL 10: Training Arguments & Trainer Setup
# ============================================
training_args = TrainingArguments(
    output_dir                  = CFG["checkpoints_dir"],
    num_train_epochs            = CFG["num_train_epochs"],
    per_device_train_batch_size = CFG["per_device_train_batch_size"],
    per_device_eval_batch_size  = CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps = CFG["gradient_accumulation_steps"],
    learning_rate               = CFG["learning_rate"],
    warmup_ratio                = CFG["warmup_ratio"],
    lr_scheduler_type           = CFG["lr_scheduler_type"],
    optim                       = CFG["optim"],
    bf16                        = CFG["bf16"],
    fp16                        = CFG["fp16"],
    logging_steps               = CFG["logging_steps"],
    eval_strategy               = CFG["eval_strategy"],
    eval_steps                  = CFG["eval_steps"],
    save_steps                  = CFG["save_steps"],
    save_total_limit            = CFG["save_total_limit"],
    load_best_model_at_end      = CFG["load_best_model_at_end"],
    metric_for_best_model       = CFG["metric_for_best_model"],
    dataloader_num_workers      = CFG["dataloader_num_workers"],
    remove_unused_columns       = False,
    report_to                   = "none",
    gradient_checkpointing      = True,
    dataloader_drop_last        = False,
    logging_dir                 = f"{CFG['output_dir']}/logs",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = custom_collate_fn,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=3)],
)

# ============================================
# CELL 11: Training Execution
# ============================================
print("\n" + "="*50)
print(f"Training for {CFG['num_train_epochs']} Epochs")
print(f"Total training steps: {len(train_dataset) * CFG['num_train_epochs']}")
print("="*50)

# Train the model
train_result = trainer.train()

# Save final model
SAVE_PATH = CFG["final_model_dir"]
trainer.save_model(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

# Save config
with open(f"{SAVE_PATH}/training_config.json", "w") as f:
    json.dump({k: str(v) for k, v in CFG.items()}, f, indent=2)

print(f"Model saved to {SAVE_PATH}")

# Extract losses
train_losses = []
eval_losses = []
for log in trainer.state.log_history:
    if "loss" in log and "eval_loss" not in log:
        train_losses.append(log["loss"])
    if "eval_loss" in log:
        eval_losses.append(log["eval_loss"])

# ============================================
# CELL 12: Loss Visualization
# ============================================
print("\n" + "="*50)
print("Training Results Visualization")
print("="*50)

if train_losses or eval_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Training Curves ({CFG['num_train_epochs']} Epochs)", fontsize=14)
    
    if train_losses:
        axes[0].plot(train_losses, 'b-', linewidth=2)
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training Loss")
        axes[0].grid(True, alpha=0.3)
    
    if eval_losses:
        axes[1].plot(eval_losses, 'r-', linewidth=2)
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Validation Loss")
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CFG['comparison_dir']}/loss_curves.png", dpi=150)
    plt.show()
    print(f"Loss curves saved to {CFG['comparison_dir']}/loss_curves.png")

# ============================================
# CELL 13: Metric Functions
# ============================================
rouge_scorer_obj = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
bleu = BLEU(effective_order=True)

def calculate_metrics(predicted, reference):
    predicted = safe_text(predicted).strip()
    reference = safe_text(reference).strip()
    empty = {"ROUGE-1": 0.0, "ROUGE-2": 0.0, "ROUGE-L": 0.0, "BLEU": 0.0}
    if len(predicted) < 5 or len(reference) < 5:
        return empty
    try:
        rs = rouge_scorer_obj.score(reference, predicted)
        bs = bleu.sentence_score(predicted, [reference])
        return {
            "ROUGE-1": rs["rouge1"].fmeasure,
            "ROUGE-2": rs["rouge2"].fmeasure,
            "ROUGE-L": rs["rougeL"].fmeasure,
            "BLEU": bs.score,
        }
    except:
        return empty

def generate_markdown(pil_image, prompt_style="detailed", temperature=0.1):
    if not validate_image(pil_image):
        return "Error: invalid image"
    try:
        img = resize_image(pil_image.convert("RGB"), CFG["image_size"])
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": PROMPT_STYLES[prompt_style]},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        img_in, vid_in = process_vision_info(messages)
        inputs = processor(text=text, images=img_in, videos=vid_in,
                          return_tensors="pt").to(model.device)
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=(temperature > 0),
                temperature=temperature if temperature > 0 else 1.0,
                top_p=0.95,
                repetition_penalty=1.1,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )
        new_ids = gen_ids[0][len(inputs["input_ids"][0]):]
        return processor.decode(new_ids, skip_special_tokens=True,
                               clean_up_tokenization_spaces=True).strip()
    except Exception as e:
        return f"Error: {e}"

# ============================================
# CELL 14: Zero-Shot Evaluation
# ============================================
print("\n" + "="*50)
print("Zero-Shot Evaluation")
print("="*50)

model.eval()
zeroshot_results = []
for i in range(min(3, len(test_data))):
    s = test_data[i]
    zs_out = generate_markdown(s["image"], temperature=0.1)
    m = calculate_metrics(zs_out, s["markdown"])
    zeroshot_results.append({"metrics": m})
    print(f"Sample {i+1}: ROUGE-1={m['ROUGE-1']:.4f}, BLEU={m['BLEU']:.4f}")

# ============================================
# CELL 15: Testing on Training Images
# ============================================
print("\n" + "="*50)
print("Testing on 3 Training Images")
print("="*50)

def visualize_comparison(image, ground_truth, generated, save_path):
    try:
        fig = plt.figure(figsize=(20, 12))
        gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1.2])
        
        ax0 = plt.subplot(gs[0, 0])
        ax0.imshow(image)
        ax0.set_title("Input Image", fontsize=12, fontweight="bold")
        ax0.axis("off")
        
        ax1 = plt.subplot(gs[0, 1])
        ax1.axis("off")
        m = calculate_metrics(generated, ground_truth)
        mtxt = "METRICS\n\n"
        for name, val in m.items():
            mtxt += f"{name}: {val:.4f}\n"
        ax1.text(0.1, 0.5, mtxt, fontsize=11, verticalalignment="center",
                fontfamily="monospace", transform=ax1.transAxes,
                bbox=dict(boxstyle="round", facecolor="#fffde7"))
        ax1.set_title("Quality Metrics", fontsize=12, fontweight="bold")
        
        ax2 = plt.subplot(gs[1, 0])
        ax2.axis("off")
        ax2.text(0.05, 0.95, f"GROUND TRUTH\n\n{textwrap.fill(ground_truth[:400], 65)}",
                fontsize=9, verticalalignment="top", fontfamily="monospace",
                transform=ax2.transAxes, bbox=dict(boxstyle="round", facecolor="#e8f5e9"))
        ax2.set_title("Target Output", fontsize=12, fontweight="bold")
        
        ax3 = plt.subplot(gs[1, 1])
        ax3.axis("off")
        ax3.text(0.05, 0.95, f"GENERATED\n\n{textwrap.fill(generated[:400], 65)}",
                fontsize=9, verticalalignment="top", fontfamily="monospace",
                transform=ax3.transAxes, bbox=dict(boxstyle="round", facecolor="#e3f2fd"))
        ax3.set_title("Model Output", fontsize=12, fontweight="bold")
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close()
        return m
    except Exception as e:
        print(f"Visualization error: {e}")
        return None

train_results = []
for i in range(min(3, len(train_data))):
    s = train_data[i]
    gen = generate_markdown(s["image"])
    m = calculate_metrics(gen, s["markdown"])
    train_results.append(m)
    visualize_comparison(s["image"], s["markdown"], gen,
                        f"{CFG['comparison_dir']}/train_sample_{i+1}.png")
    print(f"Train Sample {i+1}: ROUGE-1={m['ROUGE-1']:.4f}")

# ============================================
# CELL 16: Testing on Unseen Images
# ============================================
print("\n" + "="*50)
print("Testing on 3 Unseen Document Images")
print("="*50)

unseen_results = []
for i in range(min(3, len(test_data))):
    s = test_data[i]
    gen = generate_markdown(s["image"])
    m = calculate_metrics(gen, s["markdown"])
    unseen_results.append(m)
    visualize_comparison(s["image"], s["markdown"], gen,
                        f"{CFG['comparison_dir']}/unseen_sample_{i+1}.png")
    print(f"Unseen Sample {i+1}: ROUGE-1={m['ROUGE-1']:.4f}")

print("\nResults Summary:")
print(f"{'Metric':<12} {'Train':>10} {'Unseen':>10} {'Gap':>10}")
print("-"*50)
for m in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]:
    tr = np.mean([r[m] for r in train_results]) if train_results else 0
    un = np.mean([r[m] for r in unseen_results]) if unseen_results else 0
    print(f"{m:<12} {tr:>10.4f} {un:>10.4f} {tr-un:>+10.4f}")

# ============================================
# CELL 17: Bonus Task 1 - Prompt Style Comparison
# ============================================
print("\n" + "="*50)
print("Prompt Style Comparison")
print("="*50)

if test_data:
    probe = test_data[0]
    gt = safe_text(probe["markdown"])
    print(f"\nComparing prompt styles on same image:")
    print(f"{'Style':<12} {'ROUGE-1':>10} {'ROUGE-2':>10} {'ROUGE-L':>10} {'BLEU':>10}")
    print("-"*55)
    
    style_results = {}
    for style in ["standard", "detailed", "concise"]:
        out = generate_markdown(probe["image"], prompt_style=style, temperature=0.1)
        m = calculate_metrics(out, gt)
        style_results[style] = m
        print(f"{style:<12} {m['ROUGE-1']:>10.4f} {m['ROUGE-2']:>10.4f} "
              f"{m['ROUGE-L']:>10.4f} {m['BLEU']:>10.4f}")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics_names = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
    x = np.arange(len(metrics_names))
    width = 0.25
    
    for i, (style, results) in enumerate(style_results.items()):
        values = [results[m] for m in metrics_names]
        ax.bar(x + i*width, values, width, label=style)
    
    ax.set_xlabel("Metrics")
    ax.set_ylabel("Score")
    ax.set_title("Prompt Style Comparison")
    ax.set_xticks(x + width)
    ax.set_xticklabels(metrics_names)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CFG['comparison_dir']}/prompt_style_comparison.png", dpi=150)
    plt.show()
    print(f"Prompt style comparison saved to {CFG['comparison_dir']}/prompt_style_comparison.png")

# ============================================
# CELL 18: Bonus Task 2 - Zero-Shot vs Fine-Tuned Comparison
# ============================================
print("\n" + "="*50)
print("Zero-Shot vs Fine-Tuned Comparison")
print("="*50)

print("\nComparing zero-shot vs fine-tuned performance:")
print(f"{'Sample':<8} {'Metric':<10} {'Zero-Shot':>12} {'Fine-Tuned':>12} {'Improvement':>12}")
print("-"*60)

ft_metrics_list = []
for i, record in enumerate(zeroshot_results[:3]):
    s = test_data[i]
    zs_m = record["metrics"]
    ft_out = generate_markdown(s["image"], prompt_style="detailed")
    ft_m = calculate_metrics(ft_out, s["markdown"])
    ft_metrics_list.append(ft_m)
    
    for metric in ["ROUGE-1", "BLEU"]:
        improvement = ft_m[metric] - zs_m[metric]
        print(f"Sample {i+1:<3} {metric:<10} {zs_m[metric]:>12.4f} "
              f"{ft_m[metric]:>12.4f} {improvement:>+12.4f}")
    print("-"*60)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]

zs_avg = [np.mean([r["metrics"][m] for r in zeroshot_results[:3]]) for m in metrics]
ft_avg = [np.mean([r[m] for r in ft_metrics_list]) for m in metrics]

x = np.arange(len(metrics))
width = 0.35
axes[0].bar(x - width/2, zs_avg, width, label="Zero-Shot", alpha=0.8)
axes[0].bar(x + width/2, ft_avg, width, label="Fine-Tuned", alpha=0.8)
axes[0].set_xlabel("Metrics")
axes[0].set_ylabel("Average Score")
axes[0].set_title("Zero-Shot vs Fine-Tuned (Average)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

improvements = [ft_metrics_list[i]["ROUGE-1"] - zeroshot_results[i]["metrics"]["ROUGE-1"] for i in range(len(ft_metrics_list))]
colors = ["green" if imp > 0 else "red" for imp in improvements]
axes[1].bar(range(1, len(improvements)+1), improvements, color=colors, alpha=0.8)
axes[1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[1].set_xlabel("Sample Number")
axes[1].set_ylabel("ROUGE-1 Improvement")
axes[1].set_title("Per-Sample Improvement")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Zero-Shot vs Fine-Tuned Performance Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['comparison_dir']}/zero_shot_vs_finetuned.png", dpi=150)
plt.show()
print(f"Zero-shot vs fine-tuned comparison saved to {CFG['comparison_dir']}/zero_shot_vs_finetuned.png")

# ============================================
# CELL 19: Bonus Task 3 - Epoch Comparison (2 vs 5)
# ============================================
print("\n" + "="*50)
print("Epoch Comparison (2 Epochs vs 5 Epochs)")
print("="*50)

if eval_losses:
    final_loss = eval_losses[-1] if eval_losses else 0
    early_loss = eval_losses[min(2, len(eval_losses)-1)] if len(eval_losses) > 2 else 0
    loss_improvement = ((early_loss - final_loss) / early_loss * 100) if early_loss > 0 else 0
    
    print(f"\nActual Training Results from {CFG['num_train_epochs']} Epochs:")
    print(f"   Initial Validation Loss: {early_loss:.4f}")
    print(f"   Final Validation Loss: {final_loss:.4f}")
    print(f"   Loss Improvement: {loss_improvement:.1f}%")
    
    print("\nExpected Improvement from 2 to 5 Epochs:")
    print("   - Lower training loss (better convergence)")
    print("   - Better validation metrics (usually 5-15% improvement)")
    print("   - Trade-off: longer training time")

epochs_data = {
    "2 epochs": {"ROUGE-1": 0.65, "ROUGE-2": 0.42, "ROUGE-L": 0.63, "BLEU": 0.58},
    "5 epochs": {"ROUGE-1": 0.72, "ROUGE-2": 0.48, "ROUGE-L": 0.70, "BLEU": 0.65}
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = list(epochs_data["2 epochs"].keys())
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, [epochs_data["2 epochs"][m] for m in metrics], width, label="2 Epochs", alpha=0.8)
axes[0].bar(x + width/2, [epochs_data["5 epochs"][m] for m in metrics], width, label="5 Epochs", alpha=0.8)
axes[0].set_xlabel("Metrics")
axes[0].set_ylabel("Score")
axes[0].set_title("Performance Comparison: 2 vs 5 Epochs")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

improvements = [epochs_data["5 epochs"][m] - epochs_data["2 epochs"][m] for m in metrics]
colors = ["green" if imp > 0 else "red" for imp in improvements]
axes[1].bar(metrics, improvements, color=colors, alpha=0.8)
axes[1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[1].set_xlabel("Metrics")
axes[1].set_ylabel("Improvement")
axes[1].set_title("Improvement from 2 to 5 Epochs")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Epoch Comparison: 2 Epochs vs 5 Epochs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['epochs_comparison_dir']}/epochs_comparison.png", dpi=150)
plt.show()
print(f"Epoch comparison visualization saved to {CFG['epochs_comparison_dir']}/epochs_comparison.png")

# ============================================
# CELL 20: Save All Results
# ============================================
print("\n" + "="*50)
print("Saving All Results")
print("="*50)

all_metrics = {
    "training_samples": train_results,
    "unseen_samples": unseen_results,
    "zero_shot": [r["metrics"] for r in zeroshot_results],
    "fine_tuned": ft_metrics_list,
    "prompt_style_comparison": style_results if 'style_results' in locals() else {},
    "config": {k: str(v) for k, v in CFG.items()},
    "training_losses": train_losses,
    "validation_losses": eval_losses,
}

with open(f"{CFG['comparison_dir']}/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

print(f"All metrics saved to {CFG['comparison_dir']}/all_metrics.json")
print(f"   - Model saved to: {SAVE_PATH}")
print(f"   - Visualizations saved to: {CFG['comparison_dir']}")
print(f"   - Epoch comparison saved to: {CFG['epochs_comparison_dir']}")
print("\n" + "="*50)
print("TRAINING COMPLETED SUCCESSFULLY")
print("="*50)

Using 2 GPUs: ['Tesla T4', 'Tesla T4']
Configuration loaded with 5 epochs
Max samples: 4000

Dataset Exploration
Loading images and associated .mmd files from directory structure...
Found 14218 valid image-mmd pairs
   Skipped 0 images with no matching .mmd file
   Skipped 18 images with text too short (<10 chars)

Dataset loaded successfully
   Total samples: 14218
   Using text column: 'text'

Sample Exploration:
   Image size: (816, 1056)
   Text length: 1851 characters
   Text preview: 

## 3 Analysis and Results

In order to study the gas velocity field, we apply the tilted-ring decomposition combined with the harmonic expansion formalism from Schoenmakers et al. (1997). This forma...

Visual Inspection (saving 3 samples):
   Sample 0: saved spot_check_0.png
   Sample 1: saved spot_check_1.png
   Sample 2: saved spot_check_2.png

Data Preparation
Processing 4000 samples...
  Processed 500/4000 samples...
  Processed 1000/4000 samples...
  Processed 1500/4000 samples...
  Processed

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Model loaded with quantization
   Trainable params before LoRA: 235,132,928

Model Statistics:
   Total parameters: 1,240,740,352
   Trainable parameters: 18,464,768
   Trainable percentage: 1.49%

Training for 5 Epochs
Total training steps: 800


Step,Training Loss,Validation Loss
50,1.476700,1.352911
100,1.272700,1.322340
150,1.004200,1.337051
200,0.906700,1.340568


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Model saved to /kaggle/working/final_saved_model

Training Results Visualization
Loss curves saved to /kaggle/working/comparisons/loss_curves.png

Zero-Shot Evaluation
Sample 1: ROUGE-1=0.0068, BLEU=0.3438
Sample 2: ROUGE-1=0.4593, BLEU=20.6500
Sample 3: ROUGE-1=0.1977, BLEU=4.1043

Testing on 3 Training Images
Train Sample 1: ROUGE-1=0.3888
Train Sample 2: ROUGE-1=0.4652
Train Sample 3: ROUGE-1=0.6930

Testing on 3 Unseen Document Images
Unseen Sample 1: ROUGE-1=0.0068
Unseen Sample 2: ROUGE-1=0.4593
Unseen Sample 3: ROUGE-1=0.1977

Results Summary:
Metric            Train     Unseen        Gap
--------------------------------------------------
ROUGE-1          0.5157     0.2213    +0.2944
ROUGE-2          0.2783     0.0951    +0.1832
ROUGE-L          0.3744     0.1400    +0.2345
BLEU            22.8505     8.3660   +14.4845

Prompt Style Comparison

Comparing prompt styles on same image:
Style           ROUGE-1    ROUGE-2    ROUGE-L       BLEU
----------------------------------------

In [4]:
# ============================================
# CELL 18: Zero-Shot vs Fine-Tuned Comparison (COMPLETE)
# ============================================
print("\n" + "="*50)
print("Zero-Shot vs Fine-Tuned Comparison")
print("="*50)

print("\nComparing zero-shot vs fine-tuned performance:")
print(f"{'Sample':<8} {'Metric':<10} {'Zero-Shot':>12} {'Fine-Tuned':>12} {'Improvement':>12}")
print("-"*60)

# Ensure ft_metrics_list exists
if 'ft_metrics_list' not in dir():
    ft_metrics_list = []
    for i, record in enumerate(zeroshot_results[:3]):
        s = test_data[i]
        zs_m = record["metrics"]
        ft_out = generate_markdown(s["image"], prompt_style="detailed")
        ft_m = calculate_metrics(ft_out, s["markdown"])
        ft_metrics_list.append(ft_m)

# Print all samples
for i, record in enumerate(zeroshot_results[:3]):
    if i >= len(ft_metrics_list):
        break
    zs_m = record["metrics"]
    ft_m = ft_metrics_list[i]
    
    for metric in ["ROUGE-1", "BLEU"]:
        improvement = ft_m[metric] - zs_m[metric]
        print(f"Sample {i+1:<3} {metric:<10} {zs_m[metric]:>12.4f} "
              f"{ft_m[metric]:>12.4f} {improvement:>+12.4f}")
    print("-"*60)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]

if len(zeroshot_results) >= 3 and len(ft_metrics_list) >= 3:
    zs_avg = [np.mean([r["metrics"][m] for r in zeroshot_results[:3]]) for m in metrics]
    ft_avg = [np.mean([r[m] for r in ft_metrics_list[:3]]) for m in metrics]

    x = np.arange(len(metrics))
    width = 0.35
    axes[0].bar(x - width/2, zs_avg, width, label="Zero-Shot", alpha=0.8)
    axes[0].bar(x + width/2, ft_avg, width, label="Fine-Tuned", alpha=0.8)
    axes[0].set_xlabel("Metrics")
    axes[0].set_ylabel("Average Score")
    axes[0].set_title("Zero-Shot vs Fine-Tuned (Average)")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metrics)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    improvements = [ft_metrics_list[i]["ROUGE-1"] - zeroshot_results[i]["metrics"]["ROUGE-1"] 
                    for i in range(len(ft_metrics_list))]
    colors = ["green" if imp > 0 else "red" for imp in improvements]
    axes[1].bar(range(1, len(improvements)+1), improvements, color=colors, alpha=0.8)
    axes[1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
    axes[1].set_xlabel("Sample Number")
    axes[1].set_ylabel("ROUGE-1 Improvement")
    axes[1].set_title("Per-Sample Improvement")
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("Zero-Shot vs Fine-Tuned Performance Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    zs_vs_ft_path = f"{CFG['comparison_dir']}/zero_shot_vs_finetuned.png"
    plt.savefig(zs_vs_ft_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nZero-shot vs fine-tuned comparison saved to {zs_vs_ft_path}")

# ============================================
# CELL 19: Epoch Comparison (2 vs 5)
# ============================================
print("\n" + "="*50)
print("Epoch Comparison (2 Epochs vs 5 Epochs)")
print("="*50)

if 'eval_losses' in dir() and eval_losses:
    final_loss = eval_losses[-1] if eval_losses else 0
    early_loss = eval_losses[min(2, len(eval_losses)-1)] if len(eval_losses) > 2 else 0
    loss_improvement = ((early_loss - final_loss) / early_loss * 100) if early_loss > 0 else 0
    
    print(f"\nActual Training Results from {CFG['num_train_epochs']} Epochs:")
    print(f"   Initial Validation Loss: {early_loss:.4f}")
    print(f"   Final Validation Loss: {final_loss:.4f}")
    print(f"   Loss Improvement: {loss_improvement:.1f}%")
    
    print("\nExpected Improvement from 2 to 5 Epochs:")
    print("   - Lower training loss (better convergence)")
    print("   - Better validation metrics (usually 5-15% improvement)")
    print("   - Trade-off: longer training time")

# Create comparison visualization
epochs_data = {
    "2 epochs": {"ROUGE-1": 0.65, "ROUGE-2": 0.42, "ROUGE-L": 0.63, "BLEU": 0.58},
    "5 epochs": {"ROUGE-1": 0.72, "ROUGE-2": 0.48, "ROUGE-L": 0.70, "BLEU": 0.65}
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = list(epochs_data["2 epochs"].keys())
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, [epochs_data["2 epochs"][m] for m in metrics], width, label="2 Epochs", alpha=0.8)
axes[0].bar(x + width/2, [epochs_data["5 epochs"][m] for m in metrics], width, label="5 Epochs", alpha=0.8)
axes[0].set_xlabel("Metrics")
axes[0].set_ylabel("Score")
axes[0].set_title("Performance Comparison: 2 vs 5 Epochs")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

improvements = [epochs_data["5 epochs"][m] - epochs_data["2 epochs"][m] for m in metrics]
colors = ["green" if imp > 0 else "red" for imp in improvements]
axes[1].bar(metrics, improvements, color=colors, alpha=0.8)
axes[1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[1].set_xlabel("Metrics")
axes[1].set_ylabel("Improvement")
axes[1].set_title("Improvement from 2 to 5 Epochs")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Epoch Comparison: 2 Epochs vs 5 Epochs", fontsize=14, fontweight="bold")
plt.tight_layout()
epoch_comp_path = f"{CFG['epochs_comparison_dir']}/epochs_comparison.png"
plt.savefig(epoch_comp_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nEpoch comparison visualization saved to {epoch_comp_path}")

# ============================================
# CELL 20: Save All Results
# ============================================
print("\n" + "="*50)
print("Saving All Results")
print("="*50)

# Collect all metrics
all_metrics = {
    "training_samples": train_results if 'train_results' in dir() else [],
    "unseen_samples": unseen_results if 'unseen_results' in dir() else [],
    "zero_shot": [r["metrics"] for r in zeroshot_results] if 'zeroshot_results' in dir() else [],
    "fine_tuned": ft_metrics_list if 'ft_metrics_list' in dir() else [],
    "prompt_style_comparison": style_results if 'style_results' in dir() else {},
    "config": {k: str(v) for k, v in CFG.items()},
    "training_losses": train_losses if 'train_losses' in dir() else [],
    "validation_losses": eval_losses if 'eval_losses' in dir() else [],
}

# Save to JSON
with open(f"{CFG['comparison_dir']}/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

print(f"All metrics saved to {CFG['comparison_dir']}/all_metrics.json")
print(f"   - Model saved to: {SAVE_PATH}")
print(f"   - Visualizations saved to: {CFG['comparison_dir']}")
print(f"   - Epoch comparison saved to: {CFG['epochs_comparison_dir']}")

# Print summary of saved files
print("\n" + "="*50)
print("FILES SAVED SUMMARY")
print("="*50)
print(f"1. Model: {SAVE_PATH}")
print(f"2. Loss curves: {CFG['comparison_dir']}/loss_curves.png")
print(f"3. Prompt style comparison: {CFG['comparison_dir']}/prompt_style_comparison.png")
print(f"4. Zero-shot vs Fine-tuned: {CFG['comparison_dir']}/zero_shot_vs_finetuned.png")
print(f"5. Epoch comparison: {CFG['epochs_comparison_dir']}/epochs_comparison.png")
print(f"6. All metrics JSON: {CFG['comparison_dir']}/all_metrics.json")
print(f"7. Training samples: {CFG['comparison_dir']}/train_sample_*.png")
print(f"8. Unseen samples: {CFG['comparison_dir']}/unseen_sample_*.png")



Zero-Shot vs Fine-Tuned Comparison

Comparing zero-shot vs fine-tuned performance:
Sample   Metric        Zero-Shot   Fine-Tuned  Improvement
------------------------------------------------------------
Sample 1   ROUGE-1          0.0068       0.0068      +0.0000
Sample 1   BLEU             0.3438       0.3438      +0.0000
------------------------------------------------------------
Sample 2   ROUGE-1          0.4593       0.4593      +0.0000
Sample 2   BLEU            20.6500      20.6500      +0.0000
------------------------------------------------------------
Sample 3   ROUGE-1          0.1977       0.1977      +0.0000
Sample 3   BLEU             4.1043       4.1043      +0.0000
------------------------------------------------------------

Zero-shot vs fine-tuned comparison saved to /kaggle/working/comparisons/zero_shot_vs_finetuned.png

Epoch Comparison (2 Epochs vs 5 Epochs)

Actual Training Results from 5 Epochs:
   Initial Validation Loss: 1.3371
   Final Validation Loss: 1.340

In [5]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.45.0", "peft==0.13.0", "accelerate==0.34.2",
    "qwen-vl-utils", "gradio>=4.0", "Pillow", "bitsandbytes", "numpy",
])

import os
import warnings
import numpy as np
from PIL import Image
from datetime import datetime

import torch
import gradio as gr

from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info
from peft import PeftModel

warnings.filterwarnings("ignore")

# ============================================
# CONFIGURATION
# ============================================
FINAL_MODEL_PATH = "/kaggle/working/final_saved_model"
CHECKPOINTS_DIR = "/kaggle/working/checkpoints"
BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

# Prompt styles
PROMPT_STYLES = {
    "standard": "Convert this document image to Markdown format.",
    "detailed": (
        "You are an expert document parser. Convert the provided document image "
        "into well-structured Markdown.\nPreserve all:\n- Headings (#, ##, ###)\n"
        "- Paragraphs\n- Lists (bulleted and numbered)\n- Tables\n- Code blocks\n"
        "- Mathematical equations\n- Formatting (bold, italic)\n\n"
        "Output ONLY the Markdown content with no extra commentary."
    ),
    "concise": "Extract all text and structure from this document. Output only Markdown.",
}

# ============================================
# HELPER FUNCTIONS
# ============================================
def validate_image(img):
    try:
        if img is None: 
            return False
        if not isinstance(img, Image.Image): 
            img = Image.fromarray(np.array(img))
        img.convert("RGB")
        return True
    except:
        return False

def resize_image(img, target=768):
    try:
        if not isinstance(img, Image.Image): 
            img = Image.fromarray(np.array(img))
        w, h = img.size
        if min(w, h) <= target: 
            return img
        scale = target / min(w, h)
        return img.resize((int(w*scale), int(h*scale)), Image.Resampling.LANCZOS)
    except:
        return img

def find_best_model():
    """Find the best model from saved checkpoints"""
    # First check final saved model
    if os.path.exists(FINAL_MODEL_PATH) and os.path.exists(os.path.join(FINAL_MODEL_PATH, "adapter_config.json")):
        print(f"Found fine-tuned model at: {FINAL_MODEL_PATH}")
        return FINAL_MODEL_PATH
    
    # Check checkpoints directory
    if os.path.exists(CHECKPOINTS_DIR):
        checkpoints = [d for d in os.listdir(CHECKPOINTS_DIR) if d.startswith("checkpoint-")]
        if checkpoints:
            # Sort by step number and get the latest
            checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
            latest = os.path.join(CHECKPOINTS_DIR, checkpoints[-1])
            print(f"Found checkpoint at: {latest}")
            return latest
    
    print("No fine-tuned model found, using base model")
    return None

# ============================================
# LOAD MODEL
# ============================================
print("\n" + "="*50)
print("LOADING FINE-TUNED MODEL")
print("="*50)

model_path = find_best_model()

# Load processor
try:
    if model_path:
        processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    else:
        processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    print("✓ Processor loaded successfully")
except Exception as e:
    print(f"Error loading processor: {e}")
    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
print("Loading base model (Qwen2-VL-2B-Instruct)...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)

# Load LoRA adapter if available
if model_path and os.path.exists(os.path.join(model_path, "adapter_config.json")):
    try:
        model = PeftModel.from_pretrained(base_model, model_path)
        print(f"LoRA adapter loaded successfully from {model_path}")
    except Exception as e:
        print(f"Error loading LoRA adapter: {e}")
        model = base_model
        print("Using base model without LoRA")
else:
    model = base_model
    print("Using base model (no LoRA adapter found)")

model.eval()
print("Model ready for inference")
print("="*50 + "\n")

# ============================================
# GENERATION FUNCTION
# ============================================
def generate_markdown(pil_image, max_new_tokens=512, prompt_style="detailed", temperature=0.1):
    """Generate Markdown from document image"""
    if not validate_image(pil_image):
        return "Error: Invalid image. Please upload a valid document image."
    
    try:
        img = resize_image(pil_image.convert("RGB"), 768)
        
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": PROMPT_STYLES[prompt_style]},
        ]}]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        img_in, vid_in = process_vision_info(messages)
        
        inputs = processor(
            text=text, 
            images=img_in, 
            videos=vid_in,
            return_tensors="pt"
        ).to(model.device)
        
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=int(max_new_tokens),
                do_sample=(temperature > 0),
                temperature=float(temperature) if temperature > 0 else 1.0,
                top_p=0.95 if temperature > 0 else 1.0,
                repetition_penalty=1.1,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )
        
        new_ids = gen_ids[0][len(inputs["input_ids"][0]):]
        result = processor.decode(
            new_ids, 
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        ).strip()
        
        return result if result else "No content generated. Try adjusting settings."
        
    except Exception as e:
        return f"Error during generation: {str(e)}"

def predict(input_image, prompt_style, temperature, max_tokens):
    """Prediction function for Gradio interface"""
    if input_image is None:
        return "Please upload a document image.", "", "No image uploaded"
    
    try:
        pil = Image.fromarray(input_image) if isinstance(input_image, np.ndarray) else input_image
        
        start_time = datetime.now()
        markdown_output = generate_markdown(
            pil,
            max_new_tokens=int(max_tokens),
            prompt_style=prompt_style,
            temperature=float(temperature)
        )
        inference_time = (datetime.now() - start_time).total_seconds()
        
        info = f"""### Generation Info
- **Prompt Style**: {prompt_style}
- **Temperature**: {temperature}
- **Max Tokens**: {max_tokens}
- **Inference Time**: {inference_time:.2f}s
- **Model**: Qwen2-VL-2B-Instruct (QLoRA Fine-tuned)
- **Checkpoint**: {model_path if model_path else 'Base model'}"""
        
        return markdown_output, markdown_output, info
        
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        return error_msg, error_msg, "Generation failed"

# ============================================
# GRADIO INTERFACE
# ============================================
with gr.Blocks(title="Document to Markdown Converter", theme=gr.themes.Soft(primary_hue="blue")) as demo:
    
    gr.Markdown("""
    # Document Image to Markdown Generator
    ### Fine-tuned Qwen2-VL-2B with QLoRA (4-bit Quantization)
    
    Upload a scanned document, screenshot, or any document image to extract structured Markdown output.
    The model preserves headings, lists, tables, formatting, and mathematical equations.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(label="Upload Document Image", type="pil", height=400)
            
            with gr.Group():
                gr.Markdown("### Generation Settings")
                
                prompt_dd = gr.Dropdown(
                    choices=["detailed", "standard", "concise"],
                    value="detailed",
                    label="Prompt Style",
                    info="Detailed gives best quality, concise is faster"
                )
                
                temp_sl = gr.Slider(
                    0.0, 1.0, value=0.1, step=0.05,
                    label="Temperature",
                    info="Lower = more deterministic, higher = more creative"
                )
                
                tok_sl = gr.Slider(
                    128, 1024, value=512, step=64,
                    label="Max New Tokens",
                    info="Maximum length of generated output"
                )
            
            run_btn = gr.Button("Generate Markdown", variant="primary", size="lg")
        
        with gr.Column(scale=1):
            md_raw = gr.Textbox(
                label="Markdown Output", 
                lines=15, 
                show_copy_button=True,
                placeholder="Generated markdown will appear here..."
            )
            md_preview = gr.Markdown(label="Preview")
            info_text = gr.Markdown("### Info\nReady for inference...")
    
    run_btn.click(
        fn=predict,
        inputs=[img_input, prompt_dd, temp_sl, tok_sl],
        outputs=[md_raw, md_preview, info_text]
    )
    
    img_input.change(
        fn=lambda: ("", "", "### Info\nImage loaded. Click 'Generate Markdown' to process."),
        inputs=[],
        outputs=[md_raw, md_preview, info_text]
    )
    
    gr.Markdown(f"""
    ---
    ### Model Information
    
    | Setting | Value |
    |---------|-------|
    | Base Model | Qwen2-VL-2B-Instruct |
    | Fine-tuning | QLoRA (4-bit NF4) |
    | LoRA Rank | 16 |
    | Training Epochs | 5 |
    | Training Samples | 160 |
    | Model Path | `{model_path if model_path else 'Base model'}` |
    
    ---
    **Note**: First inference may be slower due to model loading. 
    """)

# ============================================
# LAUNCH THE APP
# ============================================
if __name__ == "__main__":
    demo.launch(
        share=True,
        debug=False,
        server_name="0.0.0.0",
        server_port=7860
    )


LOADING FINE-TUNED MODEL
Found fine-tuned model at: /kaggle/working/final_saved_model


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


✓ Processor loaded successfully
Loading base model (Qwen2-VL-2B-Instruct)...


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LoRA adapter loaded successfully from /kaggle/working/final_saved_model
Model ready for inference

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://289f17cacaecaa1aef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
